
# HIGGS Dataset — Data Preprocessing

**Project:** Machine Learning-Based Higgs Boson Signal Detection

**Dataset:** HIGGS (11 Million Records)

## 1. Introduction

This notebook prepares the complete HIGGS dataset for machine learning model training.

### Objectives
- Handle duplicate records.
- Prevent data leakage.
- Split data into training, validation, and test sets.
- Apply appropriate feature scaling.
- Save processed datasets for model training.



## 2. Environment Setup

Connect Google Drive to access the HIGGS dataset.


In [1]:

from google.colab import drive

drive.mount('/content/drive')


Mounted at /content/drive



### 2.1 Import Libraries

Import libraries required for data preprocessing.


In [2]:

import pandas as pd
import numpy as np
import os
import gc
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

print("Libraries imported successfully!")


Libraries imported successfully!



### 2.2 Configure Project Paths

Define the raw and processed dataset directories.


In [3]:

PROJECT_PATH = "/content/drive/MyDrive/Higgs-boson-signal-detection-ML"

RAW_DATA_PATH = os.path.join(
    PROJECT_PATH, "data", "raw"
)

PROCESSED_DATA_PATH = os.path.join(
    PROJECT_PATH, "data", "processed"
)

MODEL_PATH = os.path.join(
    PROJECT_PATH, "models"
)

os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)
os.makedirs(MODEL_PATH, exist_ok=True)

print("Raw data:", RAW_DATA_PATH)
print("Processed data:", PROCESSED_DATA_PATH)
print("Models:", MODEL_PATH)


Raw data: /content/drive/MyDrive/Higgs-boson-signal-detection-ML/data/raw
Processed data: /content/drive/MyDrive/Higgs-boson-signal-detection-ML/data/processed
Models: /content/drive/MyDrive/Higgs-boson-signal-detection-ML/models



## 3. Load Dataset

Load the complete HIGGS dataset using float32 to optimize memory usage.


In [4]:

dataset_files = [
    f for f in os.listdir(RAW_DATA_PATH)
    if f.lower().endswith((".csv", ".csv.gz"))
]

if len(dataset_files) != 1:
    raise ValueError(
        f"Expected one CSV dataset, found: {dataset_files}"
    )

dataset_path = os.path.join(
    RAW_DATA_PATH,
    dataset_files[0]
)

df = pd.read_csv(
    dataset_path,
    header=None,
    dtype=np.float32
)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
print(
    f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**3:.2f} GB"
)

assert df.shape == (11_000_000, 29), (
    f"Unexpected dataset shape: {df.shape}"
)


Dataset loaded successfully!
Dataset shape: (11000000, 29)
Memory usage: 1.19 GB



### 3.1 Assign Column Names

Assign meaningful names to the target variable and 28 input features.


In [5]:

column_names = [
    "target",
    "lepton_pT",
    "lepton_eta",
    "lepton_phi",
    "missing_energy_magnitude",
    "missing_energy_phi",
    "jet1_pT",
    "jet1_eta",
    "jet1_phi",
    "jet1_b_tag",
    "jet2_pT",
    "jet2_eta",
    "jet2_phi",
    "jet2_b_tag",
    "jet3_pT",
    "jet3_eta",
    "jet3_phi",
    "jet3_b_tag",
    "jet4_pT",
    "jet4_eta",
    "jet4_phi",
    "jet4_b_tag",
    "m_jj",
    "m_jjj",
    "m_lv",
    "m_jlv",
    "m_bb",
    "m_wbb",
    "m_wwbb"
]

df.columns = column_names

print("Total records:", len(df))
print("Input features:", len(df.columns) - 1)

display(df.head())


Total records: 11000000
Input features: 28


,target,lepton_pT,lepton_eta,lepton_phi,missing_energy_magnitude,missing_energy_phi,jet1_pT,jet1_eta,jet1_phi,jet1_b_tag,...,jet4_eta,jet4_phi,jet4_b_tag,m_jj,m_jjj,m_lv,m_jlv,m_bb,m_wbb,m_wwbb
0,1.0,0.869293,-0.635082,0.225690,0.327470,-0.689993,0.754202,-0.248573,-1.092064,0.000000,...,-0.010455,-0.045767,3.101961,1.353760,0.979563,0.978076,0.920005,0.721657,0.988751,0.876678
1,1.0,0.907542,0.329147,0.359412,1.497970,-0.313010,1.095531,-0.557525,-1.588230,2.173076,...,-1.138930,-0.000819,0.000000,0.302220,0.833048,0.985700,0.978098,0.779732,0.992356,0.798343
2,1.0,0.798835,1.470639,-1.635975,0.453773,0.425629,1.104875,1.282322,1.381664,0.000000,...,1.128848,0.900461,0.000000,0.909753,1.108330,0.985692,0.951331,0.803252,0.865924,0.780118
3,0.0,1.344385,-0.876626,0.935913,1.992050,0.882454,1.786066,-1.646778,-0.942383,0.000000,...,-0.678379,-1.360356,0.000000,0.946652,1.028704,0.998656,0.728281,0.869200,1.026736,0.957904
4,1.0,1.105009,0.321356,1.522401,0.882808,-1.205349,0.681466,-1.070464,-0.921871,0.000000,...,-0.373566,0.113041,0.000000,0.755856,1.361057,0.986610,0.838085,1.133295,0.872245,0.808487



## 4. Duplicate Handling

Remove exact duplicate records before data splitting to prevent duplicate leakage between datasets.


In [6]:

original_count = len(df)

df = df.drop_duplicates(
    keep="first"
).reset_index(drop=True)

removed_count = original_count - len(df)

print("Original records:", original_count)
print("Removed duplicates:", removed_count)
print("Remaining records:", len(df))
print(f"Removed percentage: {removed_count / original_count * 100:.2f}%")


Original records: 11000000
Removed duplicates: 278698
Remaining records: 10721302
Removed percentage: 2.53%



### 4.1 Verify Duplicate Removal

Confirm that no exact duplicate records remain.


In [7]:

remaining_duplicates = df.duplicated().sum()

print("Remaining duplicate records:", remaining_duplicates)

assert remaining_duplicates == 0

print("Duplicate removal verified successfully!")


Remaining duplicate records: 0
Duplicate removal verified successfully!



### 4.2 Duplicate Handling Summary

- **Original Records:** 11,000,000
- **Removed Duplicates:** 278,698 (2.53%)
- **Remaining Records:** 10,721,302

**Conclusion:** Exact duplicate records were removed before splitting to reduce duplicate leakage. The original dataset remains unchanged.



### 4.3 Additional Preprocessing Decisions

- **Missing Values:** No imputation required, as no missing values were found.
- **Categorical Encoding:** Not required, as all input features are numerical.
- **Outlier Removal:** Not applied, as extreme values may represent meaningful particle collision measurements.

**Conclusion:** Only necessary preprocessing operations will be applied to preserve important information in the HIGGS dataset.



## 5. Data Splitting

### 5.1 Separate Features and Target

Separate the 28 input features from the binary target variable.


In [8]:

X = df.drop(columns=["target"])
y = df["target"].astype("int8")

print("Features shape:", X.shape)
print("Target shape:", y.shape)


Features shape: (10721302, 28)
Target shape: (10721302,)



### 5.2 Training and Temporary Split

Split the dataset into 70% training and 30% temporary data using stratified sampling.


In [9]:

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training records:", len(X_train))
print("Temporary records:", len(X_temp))


Training records: 7504911
Temporary records: 3216391



### 5.3 Validation and Test Split

Divide the temporary dataset equally into validation and test sets.


In [10]:

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)


Training: (7504911, 28)
Validation: (1608195, 28)
Testing: (1608196, 28)



### 5.4 Verify Data Splitting

Verify dataset sizes and class proportions after stratified splitting.


In [11]:

total = len(df)

split_summary = pd.DataFrame({
    "Dataset": ["Training", "Validation", "Testing"],
    "Records": [len(y_train), len(y_val), len(y_test)],
    "Percentage": [
        len(y_train) / total * 100,
        len(y_val) / total * 100,
        len(y_test) / total * 100
    ],
    "Signal (%)": [
        y_train.mean() * 100,
        y_val.mean() * 100,
        y_test.mean() * 100
    ]
})

display(split_summary.round(2))

assert len(y_train) + len(y_val) + len(y_test) == total


,Dataset,Records,Percentage,Signal (%)
0,Training,7504911,70.0,54.37
1,Validation,1608195,15.0,54.37
2,Testing,1608196,15.0,54.37



### 5.5 Data Splitting Summary

- **Training:** 70%
- **Validation:** 15%
- **Testing:** 15%
- **Method:** Stratified Random Splitting
- **Random State:** 42

**Conclusion:** The dataset was divided into three subsets while preserving class proportions. The test set will remain untouched during model development.



## 6. Feature Scaling

### 6.1 StandardScaler

Standardize numerical features to have approximately zero mean and unit variance using training data statistics.


In [12]:

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

print("StandardScaler initialized successfully!")


StandardScaler initialized successfully!



### 6.2 Scale Training Data

Fit the scaler only on training data to prevent data leakage.


In [13]:

X_train_scaled = scaler.fit_transform(X_train)

X_train_scaled = X_train_scaled.astype(np.float32)

print("Scaled training shape:", X_train_scaled.shape)


Scaled training shape: (7504911, 28)



### 6.3 Scale Validation and Test Data

Apply the training-fitted scaler to validation and test datasets.


In [14]:

X_val_scaled = scaler.transform(X_val).astype(np.float32)

X_test_scaled = scaler.transform(X_test).astype(np.float32)

print("Scaled validation shape:", X_val_scaled.shape)
print("Scaled test shape:", X_test_scaled.shape)


Scaled validation shape: (1608195, 28)
Scaled test shape: (1608196, 28)



### 6.4 Verify Scaling

Check the mean and standard deviation of the scaled training features.


In [15]:

print("Mean of scaled training features:")
print(np.round(X_train_scaled.mean(axis=0), 3))

print("\nStandard deviation of scaled training features:")
print(np.round(X_train_scaled.std(axis=0), 3))


Mean of scaled training features:
[ 0. -0.  0. -0. -0. -0. -0. -0.  0. -0. -0.  0. -0. -0. -0.  0. -0.  0.
 -0.  0. -0.  0. -0. -0.  0.  0.  0. -0.]

Standard deviation of scaled training features:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1.]



### 6.5 Save StandardScaler

Save the fitted scaler for consistent preprocessing during model training and inference.


In [16]:

scaler_path = os.path.join(
    MODEL_PATH,
    "standard_scaler.joblib"
)

joblib.dump(scaler, scaler_path)

print("Scaler saved successfully!")
print("Path:", scaler_path)


Scaler saved successfully!
Path: /content/drive/MyDrive/Higgs-boson-signal-detection-ML/models/standard_scaler.joblib



### 6.6 Feature Scaling Summary

- **Method:** StandardScaler
- **Fit Dataset:** Training only
- **Validation/Test:** Transform only
- **Data Leakage:** Prevented during scaling

**Conclusion:** Features were standardized using training statistics to support scale-sensitive machine learning algorithms.



## 7. Save Processed Datasets

### 7.1 Create Output Directory

Create a directory to store the processed training, validation, and testing datasets.


In [17]:

os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

print("Output directory:", PROCESSED_DATA_PATH)


Output directory: /content/drive/MyDrive/Higgs-boson-signal-detection-ML/data/processed



### 7.2 Save Unscaled Datasets

Save the original numerical features and target labels for tree-based machine learning models.


In [19]:

def save_dataset(X_data, y_data, filename):
    dataset = X_data.copy()
    dataset["target"] = y_data.to_numpy()

    output_path = os.path.join(
        PROCESSED_DATA_PATH,
        filename
    )

    dataset.to_parquet(
        output_path,
        index=False
    )

    print(f"Saved: {filename} | Records: {len(dataset):,}")

    del dataset


save_dataset(X_train, y_train, "train.parquet")
save_dataset(X_val, y_val, "validation.parquet")
save_dataset(X_test, y_test, "test.parquet")


Saved: train.parquet | Records: 7,504,911
Saved: validation.parquet | Records: 1,608,195
Saved: test.parquet | Records: 1,608,196



### 7.3 Save Scaled Datasets

Save standardized numerical features and original target labels for scale-sensitive algorithms.


In [20]:

def save_scaled_dataset(X_scaled, y_data, filename):
    dataset = pd.DataFrame(
        X_scaled,
        columns=X_train.columns
    )

    dataset["target"] = y_data.to_numpy()

    output_path = os.path.join(
        PROCESSED_DATA_PATH,
        filename
    )

    dataset.to_parquet(
        output_path,
        index=False
    )

    print(f"Saved: {filename} | Records: {len(dataset):,}")

    del dataset


save_scaled_dataset(
    X_train_scaled, y_train, "train_scaled.parquet"
)

save_scaled_dataset(
    X_val_scaled, y_val, "validation_scaled.parquet"
)

save_scaled_dataset(
    X_test_scaled, y_test, "test_scaled.parquet"
)


Saved: train_scaled.parquet | Records: 7,504,911
Saved: validation_scaled.parquet | Records: 1,608,195
Saved: test_scaled.parquet | Records: 1,608,196



### 7.4 Verify Saved Datasets

Verify that all six processed dataset files were saved successfully.


In [21]:

processed_files = [
    "train.parquet",
    "validation.parquet",
    "test.parquet",
    "train_scaled.parquet",
    "validation_scaled.parquet",
    "test_scaled.parquet"
]

for filename in processed_files:
    path = os.path.join(PROCESSED_DATA_PATH, filename)

    assert os.path.exists(path), f"Missing file: {filename}"

    size_mb = os.path.getsize(path) / (1024 ** 2)

    print(f"{filename}: {size_mb:.2f} MB")

print("\nAll processed datasets saved successfully!")


train.parquet: 449.21 MB
validation.parquet: 97.46 MB
test.parquet: 97.46 MB
train_scaled.parquet: 449.21 MB
validation_scaled.parquet: 97.46 MB
test_scaled.parquet: 97.46 MB

All processed datasets saved successfully!



### 7.5 Preprocessing Summary

- Removed exact duplicate records.
- Applied stratified 70/15/15 data splitting.
- Standardized numerical features using training statistics.
- Prevented data leakage during feature scaling.
- Saved scaled and unscaled datasets in Parquet format.
- Preserved binary target labels without scaling.

**Conclusion:** The processed HIGGS datasets are ready for machine learning model training and evaluation.
